# InternVL2-2B
- Environment: envs/internvl.yaml
- Runtime: local GPU or paid Colab
- GPU/VRAM: 16GB+
- Quantization: optional
- Known issues: trust_remote_code required

In [ ]:
import subprocess
import sys
from pathlib import Path

candidate_roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path('/content/LLMComparison'),
    Path('/content/drive/MyDrive/LLMComparison'),
]

PROJECT_ROOT = next(
    (
        root
        for root in candidate_roots
        if (root / 'experiments').exists() and (root / 'src').exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Project root not found. Clone into /content/LLMComparison or mount Drive first.'
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')


In [ ]:
MODEL = 'internvl2-2b'
PRESET = 'colab_paid_mid'
DATASETS = ['hf_vqa_rad']
NUM_SAMPLES = 20
SEED = 42
MAX_FALLBACK_RATE = 1.0
STRICT_SAMPLE_VALIDATION = False
OUTPUT_DIR = PROJECT_ROOT / 'results'
RUN_NAME = 'internvl2_2b_vqa'

print('Model:', MODEL)
print('Preset:', PRESET)
print('Run name:', RUN_NAME)
print('Output dir:', OUTPUT_DIR)
print('Seed:', SEED)
print('Max fallback rate:', MAX_FALLBACK_RATE)
print('Strict sample validation:', STRICT_SAMPLE_VALIDATION)

In [ ]:
import re
import time

command = [
    sys.executable,
    str(PROJECT_ROOT / 'experiments' / 'run_unified.py'),
    '--preset', PRESET,
    '--models', MODEL,
    '--datasets', *DATASETS,
    '--num-samples', str(NUM_SAMPLES),
    '--seed', str(SEED),
    '--max-fallback-rate', str(MAX_FALLBACK_RATE),
    '--skip-inaccessible',
    '--output-dir', str(OUTPUT_DIR),
    '--run-name', RUN_NAME,
]

if STRICT_SAMPLE_VALIDATION:
    command.append('--strict-sample-validation')

print('Running command:')
print(command)

completed_samples = 0
total_samples = None
start_ts = time.time()

process = subprocess.Popen(
    command,
    cwd=PROJECT_ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    text_line = line.rstrip()

    sample_match = re.search(r'Progress model=.* sample=(\d+)/(\d+)', text_line)
    if sample_match:
        completed_samples = int(sample_match.group(1))
        total_samples = int(sample_match.group(2))

    elapsed = time.time() - start_ts
    elapsed_min = elapsed / 60.0

    if total_samples and completed_samples > 0:
        avg_per_sample = elapsed / completed_samples
        remaining_samples = max(0, total_samples - completed_samples)
        eta_sec = avg_per_sample * remaining_samples
        eta_min = eta_sec / 60.0
        print(
            f'[sample {completed_samples}/{total_samples}] elapsed={elapsed_min:.1f}m eta~{eta_min:.1f}m | {text_line}'
        )
    else:
        print(f'[sample ?/?] elapsed={elapsed_min:.1f}m eta~unknown | {text_line}')

return_code = process.wait()
total_elapsed_min = (time.time() - start_ts) / 60.0
print(f'Finished with code={return_code} in {total_elapsed_min:.1f}m')

if return_code != 0:
    raise RuntimeError(f'run_unified failed with exit code {return_code}')

In [ ]:
import json
import pandas as pd

run_dir = OUTPUT_DIR / RUN_NAME
aggregate_path = run_dir / 'aggregate_metrics.json'
stats_path = run_dir / 'stats.json'
sample_metrics_path = run_dir / 'sample_metrics.jsonl'
predictions_path = run_dir / 'predictions.jsonl'
errors_path = run_dir / 'errors.jsonl'

print('Run directory:', run_dir)

if aggregate_path.exists():
    aggregate_payload = json.loads(aggregate_path.read_text())
    aggregate_rows = [
        {'model_name': model_name, **metrics}
        for model_name, metrics in aggregate_payload.get('metrics', {}).items()
    ]
    aggregate_df = pd.DataFrame(aggregate_rows)
    print('Aggregate metrics:')
    display(aggregate_df.T if not aggregate_df.empty else aggregate_df)

    fallback_columns = [c for c in aggregate_df.columns if c.endswith('_fallback_used_mean')]
    if fallback_columns:
        print('Fallback usage summary (mean per model):')
        display(aggregate_df[['model_name', *fallback_columns]])
else:
    print('aggregate_metrics.json not found yet.')

if stats_path.exists():
    stats_payload = json.loads(stats_path.read_text())
    print('Statistics:')
    display(pd.json_normalize(stats_payload.get('statistics', {})))

    fallback_rates = stats_payload.get('statistics', {}).get('_meta_fallback_rates', {})
    if fallback_rates:
        print('Fallback rates from stats metadata:')
        display(pd.DataFrame([fallback_rates]))
else:
    print('stats.json not found yet.')

if sample_metrics_path.exists():
    sample_rows = [json.loads(line) for line in sample_metrics_path.read_text().splitlines() if line.strip()]
    sample_df = pd.DataFrame(sample_rows)
    print('Sample metrics preview:')
    if not sample_df.empty:
        metric_columns = [
            column
            for column in sample_df.columns
            if column not in {'sample_id', 'dataset_name', 'model_name', 'task', 'timestamp'}
        ]
        display(sample_df[['sample_id', 'dataset_name', 'model_name', 'task', *metric_columns]].head(20))
        print('Metric availability:')
        display(sample_df[metric_columns].notna().sum().to_frame('non_null_count').T if metric_columns else sample_df.head(0))

        sample_fallback_columns = [c for c in metric_columns if c.endswith('_fallback_used')]
        if sample_fallback_columns:
            print('Sample fallback flags (first 20 rows):')
            display(sample_df[['sample_id', 'dataset_name', 'model_name', 'task', *sample_fallback_columns]].head(20))
    else:
        print('sample_metrics.jsonl is empty.')
else:
    print('sample_metrics.jsonl not found yet.')

if predictions_path.exists():
    print('Predictions file:', predictions_path)
if errors_path.exists():
    print('Errors file:', errors_path)
    error_rows = [json.loads(line) for line in errors_path.read_text().splitlines() if line.strip()]
    if error_rows:
        error_df = pd.DataFrame(error_rows)
        print('Error type counts:')
        display(error_df.groupby('error_type').size().to_frame('count').T)